Ejercicio 4: 
Bacterias: A la caza de proteínas de resistenciaDescripción del problema
Analizar el plásmido pT181 de Staphylococcus aureus (código de acceso NCBI NC_001371).   
Descargar el registro en formato GenBank (gb) mediante Entrez.   
Recorrer todas las características de tipo "CDS" (Coding Sequence).   
Extraer la secuencia de ADN de la región codificante, traducirla a proteína e imprimir los detalles de la proteína resultante.

In [8]:
import warnings
from Bio import Entrez, SeqIO
from Bio.Data.CodonTable import TranslationError

Entrez.email = "lucia.cruz104@alu.ulpgc.es"

id_organismo = "NC_005861.1"

print("=== EJERCICIO 4: ANÁLISIS DE CDS Y PROTEÍNAS EN GENOMA COMPLETO ===")

handle = Entrez.efetch(
    db="nucleotide",
    id=id_organismo,
    rettype="gbwithparts",
    retmode="text"
)
registro = SeqIO.read(handle, "genbank")
handle.close()

print(f"Organismo: {registro.description}")
print(f"Longitud del genoma: {len(registro.seq)} pb\n")

contador_cds = 0
coincidencias_exactas = 0

# Desactivar temporalmente warnings de codones parciales en consola para salida limpia
with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    for feature in registro.features:
        if feature.type == "CDS":
            contador_cds += 1

            nombre_gen = feature.qualifiers.get("gene", ["N/A"])[0]
            producto = feature.qualifiers.get("product", ["Desconocido"])[0]
            traduccion_oficial = feature.qualifiers.get("translation", ["N/A"])[0]
            es_pseudogen = "pseudo" in feature.qualifiers or "pseudogene" in feature.qualifiers

            # 1. Traducción nativa directa desde el SeqFeature
            try:
                # feature.translate aplica hebra, join, codon_start y tabla automáticamente
                proteina_obj = feature.translate(registro.seq, cds=True)
                proteina_calculada = str(proteina_obj)
                motivo_fallo = ""
            except TranslationError as e:
                # Fallback si el CDS es parcial o tiene un codón de inicio no estándar/incompleto
                proteina_obj = feature.translate(registro.seq, cds=False, to_stop=True)
                proteina_calculada = str(proteina_obj)
                motivo_fallo = f" (CDS incompleto/parcial: {e})"
            except Exception as e:
                proteina_calculada = ""
                motivo_fallo = f" (Error: {e})"

            coincide = (proteina_calculada == traduccion_oficial)
            if coincide:
                coincidencias_exactas += 1

            # Imprimir solo un resumen formateado
            print(f"--- CDS #{contador_cds} ---")
            print(f"Gen: {nombre_gen} | Producto: {producto}")
            print(f"Coordenadas: {feature.location}")
            print(f"Coincide con NCBI oficial: {coincide}{motivo_fallo}")
            if es_pseudogen:
                print("Nota: Este CDS está marcado como Pseudogén/Incompleto en NCBI.")
            print("-" * 60)

print(f"\nResumen Final:")
print(f"Total CDS analizados: {contador_cds}")
print(f"Coincidencias exactas: {coincidencias_exactas} de {contador_cds} ({ (coincidencias_exactas/contador_cds)*100:.2f}% )")

=== EJERCICIO 4: ANÁLISIS DE CDS Y PROTEÍNAS EN GENOMA COMPLETO ===
Organismo: Parachlamydia-related symbiont UWE25, complete genome
Longitud del genoma: 2414465 pb

--- CDS #1 ---
Gen: N/A | Producto: type II toxin-antitoxin system YafQ family toxin
Coordenadas: [391:610](+)
Coincide con NCBI oficial: True
------------------------------------------------------------
--- CDS #2 ---
Gen: N/A | Producto: type II toxin-antitoxin system RelE/ParE family toxin
Coordenadas: [545:746](+)
Coincide con NCBI oficial: True
------------------------------------------------------------
--- CDS #3 ---
Gen: N/A | Producto: hypothetical protein
Coordenadas: [1054:4024](+)
Coincide con NCBI oficial: True
------------------------------------------------------------
--- CDS #4 ---
Gen: N/A | Producto: hypothetical protein
Coordenadas: [4406:5840](+)
Coincide con NCBI oficial: True
------------------------------------------------------------
--- CDS #5 ---
Gen: N/A | Producto: phosphoketolase family protei